# 50 · Feature / ML — Feast, the feature store

**A feature store solves one specific, expensive problem: train/serve skew.** A model is *trained*
on features computed one way (a batch job over historical data) and then *served* features computed
another way (a low-latency lookup at request time). When those two paths drift — a different
aggregation, a different null-fill, a subtly different definition — the model sees inputs at serving
time that it never saw in training, and its quality quietly rots. **Feast makes the two paths share
one definition.** You declare a feature *once*; Feast serves it to training (offline, point-in-time
correct) and to inference (online, single-digit-millisecond) from that same declaration.

That is the whole thesis of a feature store:

> **Define a feature once. Serve it to training and to inference from the same definition.
> No skew, because there is only one definition.**

This lab wires Feast the most reliable way for a homelab — all-Postgres offline, Valkey online:

| plane | backing store | what it holds | reached by |
|-------|---------------|---------------|------------|
| **registry** | Postgres (`feast` DB, `registry_type: sql`) | the feature *definitions* — entities, feature views, feature services | every Feast call, cached `cache_ttl_seconds` |
| **offline** | Postgres (`feast` DB source tables) | the *historical* rows, with an `event_timestamp` per row | `get_historical_features` — batch, point-in-time |
| **online** | Valkey (Redis-compatible) | the *latest* materialized feature vector per entity | `get_online_features` — one keyed lookup, low-latency |

The registry and offline store are the **same** Postgres (`weyland-postgres`, `feast` database); the
online store is **Valkey** in `data-mesh`. Two feature views exercise the two halves of the pattern:
`track_audio_features` (static — the serving/online half) and `state_health_risk` (time-varying — the
point-in-time/offline half).

> **Read-only.** This notebook only *retrieves*. It connects to the **existing** registry that Dagster
> already `feast apply`-ed and materialized — it never `apply`s, never materializes, never writes a
> feature definition. (Historical retrieval does stage a short-lived entity table in the offline
> Postgres and drop it again; that is how a point-in-time join works, and it touches no feature data.)

## Setup

The `feast` client is **not** in the singleuser base image (which ships `polars`, `s3fs`, `pyarrow`,
`duckdb`, `fastavro`), so we install it here with the two backend extras this lab uses — `postgres`
(registry + offline) and `redis` (online, Valkey speaks the Redis protocol). `polars` — used to render
result frames exactly as in notebooks `20`/`22` — already ships in the image; `pandas` arrives as a
Feast dependency and is what `to_df()` returns.

In [1]:
%pip install -q "feast[postgres,redis]"


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connect — point at the live registry, write no secret to disk

A Feast client is configured by a `feature_store.yaml`. Rather than commit one (it would have to carry
a database password), we **write it at runtime from the environment**. The committed cell contains only
in-cluster DNS defaults via `os.environ.get(...)` and the one secret via `os.environ["WEYLAND_PG_PASSWORD"]`
— nothing sensitive is ever stored in the notebook. A validation run overrides `FEAST_PG_HOST` /
`FEAST_REDIS` via env without editing anything.

The config mirrors the live one the Dagster pipeline uses (`feast_repo/feature_store.yaml`):

- **registry** — `registry_type: sql`, the `feast` DB on `weyland-postgres`. `sslmode=disable` is
  **required**: Feast's psycopg3 driver asks for SSL by default, but `weyland-postgres` has Postgres-level
  TLS *off* (Istio mTLS secures the hop instead). Without the flag: *"server does not support SSL, but SSL
  was required."*
- **offline_store** — `postgres`, same host/DB; this is where the historical source tables live.
- **online_store** — `redis`, pointed at Valkey in `data-mesh`.

We open the store with `FeatureStore(repo_path=".")`. Because the registry is `sql`, it reads the
definitions **straight from Postgres** — there is no local `definitions.py` to apply and we do not apply one.

In [2]:
import os
import pathlib
import yaml

PG_HOST = os.environ.get("FEAST_PG_HOST", "weyland-postgres.weyland.svc.cluster.local")
REDIS   = os.environ.get("FEAST_REDIS", "valkey.data-mesh.svc.cluster.local:6379")
PG_PW   = os.environ["WEYLAND_PG_PASSWORD"]   # the ONLY secret, straight from the environment

cfg = {
    "project": "weyland",
    "provider": "local",
    "registry": {
        "registry_type": "sql",
        "path": f"postgresql://weyland:{PG_PW}@{PG_HOST}:5432/feast?sslmode=disable",
        "cache_ttl_seconds": 60,
    },
    "offline_store": {
        "type": "postgres",
        "host": PG_HOST, "port": 5432, "database": "feast", "db_schema": "public",
        "user": "weyland", "password": PG_PW, "sslmode": "disable",
    },
    "online_store": {"type": "redis", "connection_string": REDIS},
    "entity_key_serialization_version": 2,
}
# write the config into the working dir at runtime (with the password) — NOT committed
pathlib.Path("feature_store.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))

from feast import FeatureStore
store = FeatureStore(repo_path=".")
print("connected to Feast project :", store.project)   # proves the registry is reachable
print("registry backend           : sql (Postgres, feast DB)")
print("online backend             : redis (Valkey)")

connected to Feast project : weyland
registry backend           : sql (Postgres, feast DB)
online backend             : redis (Valkey)


/usr/local/lib/python3.12/site-packages/feast/repo_config.py:531: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(


## Discover — what does the registry know?

The registry is self-describing. Three list calls read the definitions Dagster already registered — the
**entities** (the keys features hang off), the **feature views** (grouped, versioned feature
definitions), and the **feature services** (named bundles a model requests as a unit). Nothing here is
assumed; every name used later in this notebook is one this cell prints.

In [3]:
import polars as pl

entities = store.list_entities()
feature_views = store.list_feature_views()
feature_services = store.list_feature_services()

ent_df = pl.DataFrame(
    [{"entity": e.name, "join_key": e.join_key} for e in entities]
)
print(f"{ent_df.height} entities")
ent_df

The `path` of the `RegistryConfig` starts with a plain `postgresql` string. We are updating this to `postgresql+psycopg` to ensure that the `psycopg3` driver is used by `sqlalchemy`. If you want to use `psycopg2` pass `postgresql+psycopg2` explicitly to `path`. To silence this warning, pass `postgresql+psycopg` explicitly to `path`.


2 entities


entity,join_key
str,str
"""track""","""track_id"""
"""state""","""state"""


Two entities: **`track`** (join key `track_id`) and **`state`** (join key `state`). Each feature view
hangs its features off exactly one of these keys — that key is what you pass at retrieval time to say
*which* rows you want.

In [4]:
fv_df = pl.DataFrame([
    {
        "feature_view": fv.name,
        "entities": ", ".join(fv.entities),
        "n_features": len(fv.features),
        "features": ", ".join(f.name for f in fv.features),
        "online": fv.online,
        "ttl_days": fv.ttl.days if fv.ttl else None,
    }
    for fv in feature_views
])
with pl.Config(fmt_str_lengths=120, tbl_width_chars=160):
    print(f"{fv_df.height} feature views")
    display(fv_df)

2 feature views


feature_view,entities,n_features,features,online,ttl_days
str,str,i64,str,bool,i64
"""state_health_risk""","""state""",4,"""diabetes_pct, asthma_pct, copd_pct, depression_pct""",true,36500
"""track_audio_features""","""track""",11,"""danceability, energy, key, loudness, mode, speechiness, acousticness, instrumentalness, liveness, valence, tempo""",true,36500


The two feature views are the two halves of the pattern:

- **`track_audio_features`** (entity `track`) — the 11 Spotify audio features (`danceability` … `tempo`).
  A track's audio doesn't change, so it carries a synthetic `event_timestamp` and a ~century TTL: it is
  the **static, online-serving** half — what a recommender looks up per track at request time.
- **`state_health_risk`** (entity `state`) — crude-prevalence % of four chronic conditions per US state,
  with a **real** `event_timestamp` (the BRFSS survey year). It is the **time-varying, point-in-time**
  half — the value depends on *when* you ask.

In [5]:
fs_df = pl.DataFrame([
    {
        "feature_service": fs.name,
        "feature_views": ", ".join(p.name for p in fs.feature_view_projections),
        "n_features": sum(len(p.features) for p in fs.feature_view_projections),
    }
    for fs in feature_services
])
print(f"{fs_df.height} feature services")
fs_df

2 feature services


feature_service,feature_views,n_features
str,str,i64
"""recommender_v1""","""track_audio_features""",11
"""health_v1""","""state_health_risk""",4


A **feature service** is the unit a model actually requests — `recommender_v1` bundles
`track_audio_features`, `health_v1` bundles `state_health_risk`. This is the linchpin of train/serve
parity: the *same* service object is handed to `get_historical_features` (to build a training set) and
to `get_online_features` (to serve inference), so both paths draw the identical feature set from the
identical definition. We use exactly that below — one `svc`, both calls.

## Find real entity keys — never guess a key

An online lookup for a key that doesn't exist doesn't error — it returns **nulls, silently**. So before
retrieving anything we discover keys that genuinely exist, straight from the offline source table in the
`feast` DB. We reach Postgres with `psycopg` over the same in-cluster, mTLS-secured hop Feast uses
(`sslmode=disable` for the same reason). These `track_id`s are real rows — the retrieval that follows is
therefore a genuine hit, not an accidental all-null.

In [6]:
import psycopg

pg = psycopg.connect(host=PG_HOST, port=5432, user="weyland", password=PG_PW,
                     dbname="feast", sslmode="disable")
cur = pg.cursor()

cur.execute("SELECT count(*) FROM track_audio_features")
n_tracks = cur.fetchone()[0]
cur.execute("SELECT track_id FROM track_audio_features ORDER BY track_id LIMIT 5")
track_ids = [r[0] for r in cur.fetchall()]
print(f"track_audio_features offline rows : {n_tracks}")
print("sample real track_id keys         :", track_ids)

track_audio_features offline rows : 89741
sample real track_id keys         : ['0000vdREvCVMxbQTkS888c', '000CC8EParg64OmTxVnZ0p', '000Iz0K615UepwSJ5z2RE5', '000qpdoc97IMTBvF8gwcpy', '000RDCYioLteXcutOjeweY']


## Online retrieval — the serving path (from Valkey)

This is the path an inference service walks: hand Feast a **feature service** and a list of **entity
rows** (just the keys), get back the latest materialized feature vector per key from Valkey — one keyed
lookup, no scan, no join. This is what a recommender would call to turn a `track_id` into the audio
vector its model consumes.

We pull the `recommender_v1` service by name and use it directly. Because the online store holds only
the *latest* materialized value per entity, the result has no `event_timestamp` — it is a point-in-time
*now* answer. We assert the vector came back non-null (guarding against the silent-null trap) before
trusting it.

In [7]:
svc = store.get_feature_service("recommender_v1")   # the SAME object used for training below

online = store.get_online_features(
    features=svc,
    entity_rows=[{"track_id": t} for t in track_ids],
).to_df()

feat_cols = [c for c in online.columns if c != "track_id"]
non_null = int(online[feat_cols].notna().all(axis=1).sum())
print(f"online lookup: {len(online)} keys, {non_null} with a full non-null feature vector")
if non_null == 0:
    print("WARNING: all-null — keys exist offline but were not materialized to Valkey; "
          "run `feast materialize` in the dagster pod (this notebook is read-only and does not).")
pl.from_pandas(online).select(["track_id", "danceability", "energy", "valence", "tempo", "acousticness"])

online lookup: 5 keys, 5 with a full non-null feature vector


/usr/local/lib/python3.12/site-packages/feast/infra/key_encoding_utils.py:146: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blob/master/docs/how-to-guides/entity-reserialization-of-from-v2-to-v3.md
  warnings.warn(


track_id,danceability,energy,valence,tempo,acousticness
str,f64,f64,f64,f64,f64
"""0000vdREvCVMxbQTkS888c""",0.91,0.374,0.432,104.042,0.0757
"""000CC8EParg64OmTxVnZ0p""",0.269,0.516,0.341,178.173996,0.406
"""000Iz0K615UepwSJ5z2RE5""",0.686,0.56,0.108,119.997002,0.00114
"""000qpdoc97IMTBvF8gwcpy""",0.519,0.431,0.234,129.970993,0.000964
"""000RDCYioLteXcutOjeweY""",0.679,0.77,0.839,161.720993,0.0583


Those are real audio vectors, served by key from Valkey — exactly the shape and latency an online
recommender needs. The features came out of the `recommender_v1` service, so whatever a model was
*trained* on through that service is byte-for-byte what it is *served* here.

## Historical retrieval — the training path (point-in-time, from Postgres)

The training path is different in one deep way: it must be **point-in-time correct**. To build a
training set you supply an **entity dataframe** — the keys you care about *and the timestamp as of which
you want their features*. Feast joins each row to the feature value whose `event_timestamp` is the
latest one **at or before** that timestamp, and never a later one. That "never a later one" is the
point: it prevents **label leakage**, where a model accidentally trains on a feature value that, in
production, would not have existed yet when the prediction was made.

We pass the **same `svc`** — so the columns of this training frame are identical to the online vector
above. For `track_audio_features` (static, stamped 2020-01-01) an as-of-*now* entity df simply returns
that stable vector, now carrying the `event_timestamp` the join resolved.

In [8]:
import pandas as pd

entity_df = pd.DataFrame({
    "track_id": track_ids,
    "event_timestamp": [pd.Timestamp.now(tz="UTC")] * len(track_ids),
})
hist = store.get_historical_features(entity_df=entity_df, features=svc).to_df()
print(f"historical (point-in-time) join: {len(hist)} rows, columns include the resolved event_timestamp")
pl.from_pandas(hist).select(
    ["track_id", "event_timestamp", "danceability", "energy", "valence", "tempo"]
)

historical (point-in-time) join: 5 rows, columns include the resolved event_timestamp


track_id,event_timestamp,danceability,energy,valence,tempo
str,datetime[μs],f32,f32,f32,f32
"""0000vdREvCVMxbQTkS888c""",2026-09-02 13:44:27.167838,0.91,0.374,0.432,104.042
"""000CC8EParg64OmTxVnZ0p""",2026-09-02 13:44:27.167838,0.269,0.516,0.341,178.173996
"""000Iz0K615UepwSJ5z2RE5""",2026-09-02 13:44:27.167838,0.686,0.56,0.108,119.997002
"""000qpdoc97IMTBvF8gwcpy""",2026-09-02 13:44:27.167838,0.519,0.431,0.234,129.970993
"""000RDCYioLteXcutOjeweY""",2026-09-02 13:44:27.167838,0.679,0.77,0.839,161.720993


## Point-in-time made visible — the time-varying view

The static view can't *show* point-in-time correctness (its value is the same at every timestamp). The
`state_health_risk` view can: its `event_timestamp` is the real BRFSS survey year, so the same state has
a **different** feature vector depending on the as-of timestamp. We first discover, for one real state,
which years exist — then ask for that state's features as of two different years and watch the answer
change.

In [9]:
# discover a state that has >= 2 survey years, and the years it has
cur.execute("""
    SELECT state, count(*) AS n_years
    FROM state_health_risk
    GROUP BY state
    HAVING count(*) >= 2
    ORDER BY n_years DESC, state
    LIMIT 1
""")
state, n_years = cur.fetchone()
cur.execute(
    "SELECT DISTINCT EXTRACT(YEAR FROM event_timestamp)::int AS y "
    "FROM state_health_risk WHERE state = %s ORDER BY y",
    (state,),
)
years = [r[0] for r in cur.fetchall()]
print(f"chosen state : {state!r} with {n_years} survey years -> {years}")

chosen state : 'AK' with 14 survey years -> [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


**Online** first — the serving path returns this state's *latest* materialized risk vector (Valkey keeps
only the most recent), via the `health_v1` service:

In [10]:
health_svc = store.get_feature_service("health_v1")
state_online = store.get_online_features(
    features=health_svc,
    entity_rows=[{"state": state}],
).to_df()
print(f"online (latest) risk vector for {state!r}:")
pl.from_pandas(state_online)

online (latest) risk vector for 'AK':


/usr/local/lib/python3.12/site-packages/feast/infra/key_encoding_utils.py:146: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blob/master/docs/how-to-guides/entity-reserialization-of-from-v2-to-v3.md
  warnings.warn(


state,asthma_pct,diabetes_pct,depression_pct,copd_pct
str,f64,f64,f64,f64
"""AK""",13.6,9.4,21.9,6.1


**Historical** now — the same state at its **earliest** and **latest** survey year (as of mid-year each).
Point-in-time correctness means each row resolves to the survey taken at-or-before that as-of timestamp,
so the two rows carry the two different years' values — no leakage from the later year into the earlier
row:

In [11]:
asof_years = [years[0], years[-1]]
state_entity_df = pd.DataFrame({
    "state": [state, state],
    "event_timestamp": [pd.Timestamp(f"{y}-07-01", tz="UTC") for y in asof_years],
})
state_hist = store.get_historical_features(
    entity_df=state_entity_df, features=health_svc,
).to_df()
print(f"point-in-time join for {state!r} as of mid-{asof_years[0]} and mid-{asof_years[1]}:")
pl.from_pandas(state_hist).sort("event_timestamp")

point-in-time join for 'AK' as of mid-2011 and mid-2024:


state,event_timestamp,diabetes_pct,asthma_pct,copd_pct,depression_pct
str,datetime[μs],f32,f32,f32,f32
"""AK""",2011-07-01 00:00:00,7.9,11.1,5.2,16.5
"""AK""",2024-07-01 00:00:00,9.4,13.6,6.1,21.9


Read those two rows side by side: **same state, different as-of timestamp, different feature values.**
The earlier row could not have seen the later survey — which is exactly the guarantee a training set
needs and exactly what an ordinary `JOIN ... ON state` would silently violate by attaching whichever
row it happened to match. That guarantee, applied automatically across a whole entity dataframe, is why
`get_historical_features` exists.

In [12]:
# close the psycopg connection used only for key/year discovery (Feast manages its own pools)
cur.close()
pg.close()
print("discovery connection closed — no writes were made to any feature store")

discovery connection closed — no writes were made to any feature store


## When to reach for Feast

**Reach for a feature store when a model is trained in one place and served in another** — which is
almost every real ML system:

- **You have train/serve skew, or want to prevent it** — the core case. Define the feature once; let
  Feast serve it to both `get_historical_features` (training) and `get_online_features` (inference) from
  the same feature service, and the two paths cannot drift.
- **Your training set must be point-in-time correct** — time-varying features (prices, risk scores,
  rolling counts) where an ordinary join would leak future values into past rows. `get_historical_features`
  does the as-of join for you across the whole entity dataframe.
- **You need low-latency features at request time** — the online store turns a `track_id` into its
  vector in one keyed Valkey lookup, no scan, no recompute.
- **Features are shared across models/teams** — the registry is one governed catalog of definitions;
  a feature is declared and materialized once and reused, not re-implemented per model.

**Reach for something else when the feature store's guarantees aren't the problem you have:**

| you want to… | use | in this library |
|--------------|-----|-----------------|
| serve one definition to both training and inference, point-in-time correct | **Feast** | this notebook, `50` |
| build the curated tables features are computed *from* | **dbt marts** over Trino | `40` |
| join across engines ad hoc, no materialization | **Trino** federation | `20` |
| a fast point read on one specific store | that store's **native client** | `22` |
| similarity search over embeddings | a **vector store** | `30` / `31` / `32` |

> Feast is the layer between the curated data (the dbt marts in `40`) and the model. It doesn't compute
> features — it *governs the definition* of a feature and *serves it two ways without skew*. When the
> expensive failure you're avoiding is "the model saw something at serving time it never saw in
> training," that is precisely the failure a feature store is built to remove.